In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
# ==============================================================================
# NOTEBOOK 1: DATA INGESTION, TRANSFORMATION, AND EXPERIMENTAL CONFIGURATION
# ==============================================================================

import pandas as pd
import numpy as np
import json
import os
import glob

print("--- Step 1: Loading Canonical Crowdstorming Dataset ---")
# Direct verified raw GitHub mirror of the pristine Silberzahn dataset
DATA_URL = "https://raw.githubusercontent.com/mathewzilla/redcard/master/data/CrowdstormingDataJuly1st.csv"

print(f"Downloading dataset from verified GitHub mirror...")
df_raw = pd.read_csv(DATA_URL)
print(f"✅ Loaded Pristine Dataset: {df_raw.shape[0]:,} rows by {df_raw.shape[1]} columns.")
# Ensure destination directory exists
os.makedirs("/kaggle/working/data", exist_ok=True)
RAW_DATA_PATH = "/kaggle/working/data/crowdstorming_raw.csv"
df_raw.to_csv(RAW_DATA_PATH, index=False)

--- Step 1: Loading Canonical Crowdstorming Dataset ---
✅ Loaded Pristine Dataset: 146,028 rows by 28 columns.


In [4]:
print("\n--- Step 2: Constructing Null Permutation Dataset (Placebo Control) ---")
np.random.seed(42)
df_null = df_raw.copy()

unique_players = df_null[['playerShort', 'rater1', 'rater2']].drop_duplicates().copy()
shuffled_indices = np.random.permutation(len(unique_players))

unique_players['shuffled_rater1'] = unique_players['rater1'].iloc[shuffled_indices].values
unique_players['shuffled_rater2'] = unique_players['rater2'].iloc[shuffled_indices].values

player_map_1 = dict(zip(unique_players['playerShort'], unique_players['shuffled_rater1']))
player_map_2 = dict(zip(unique_players['playerShort'], unique_players['shuffled_rater2']))

df_null['rater1'] = df_null['playerShort'].map(player_map_1)
df_null['rater2'] = df_null['playerShort'].map(player_map_2)

NULL_DATA_PATH = "/kaggle/working/data/crowdstorming_null_permuted.csv"
df_null.to_csv(NULL_DATA_PATH, index=False)
print(f"✅ Null Permuted Dataset exported to {NULL_DATA_PATH} (True Population Effect = 0.0)")

print("\n--- Step 3: Constructing Semantic Obfuscation Dataset (Disguise Control) ---")
rename_map = {
    'playerShort': 'applicant_id',
    'player': 'applicant_name',
    'club': 'employer_firm',
    'leagueCountry': 'financial_jurisdiction',
    'birthday': 'account_established_date',
    'height': 'credit_score_index',
    'weight': 'annual_income_bracket',
    'position': 'employment_specialization',
    'games': 'total_applications_reviewed',
    'victories': 'prior_approved_loans',
    'ties': 'prior_conditional_loans',
    'defeats': 'prior_denied_loans',
    'goals': 'repayment_performance_score',
    'yellowCards': 'formal_inquiry_notices',
    'yellowReds': 'elevated_risk_warnings',
    'redCards': 'loan_application_rejections',
    'photoID': 'identification_dossier_id',
    'rater1': 'risk_evaluator_alpha_score',
    'rater2': 'risk_evaluator_beta_score',
    'refNum': 'loan_officer_id',
    'refCountry': 'loan_officer_jurisdiction',
    'Alpha_3': 'jurisdiction_iso_code',
    'meanIAT': 'regional_implicit_bias_index',
    'nIAT': 'regional_iat_respondent_count',
    'seIAT': 'regional_iat_standard_error',
    'meanExp': 'regional_explicit_bias_index',
    'nExp': 'regional_exp_respondent_count',
    'seExp': 'regional_exp_standard_error'
}

df_obfuscated = df_raw.rename(columns=rename_map)
OBFUSCATED_DATA_PATH = "/kaggle/working/data/credit_risk_obfuscated.csv"
df_obfuscated.to_csv(OBFUSCATED_DATA_PATH, index=False)
print(f"✅ Semantic Obfuscation Dataset exported to {OBFUSCATED_DATA_PATH}")

print("\n--- Step 4: Building Dynamic Data Dictionaries ---")
def create_data_dictionary(dataframe, domain_description):
    lines = [domain_description]
    lines.append(f"Dimensions: {dataframe.shape[0]:,} records across {dataframe.shape[1]} variables.\n")
    lines.append("Variables Schema:")
    for col in dataframe.columns:
        dtype = str(dataframe[col].dtype)
        n_miss = dataframe[col].isnull().sum()
        pct_miss = (n_miss / len(dataframe)) * 100
        if pd.api.types.is_numeric_dtype(dataframe[col]):
            s = dataframe[col].describe()
            summary = f"Numeric | Min: {s['min']:.2f}, Mean: {s['mean']:.2f}, Max: {s['max']:.2f}"
        else:
            top_vals = dataframe[col].value_counts().head(3).index.tolist()
            summary = f"Categorical | Dominant classes: {top_vals}"
        lines.append(f"  * `{col}` ({dtype}) - {pct_miss:.1f}% missing. {summary}")
    return "\n".join(lines)

SOCCER_DICT = create_data_dictionary(
    df_raw, 
    "Domain: European Soccer Disciplinary Data (England, Germany, France, Spain; 2012-2013 Season).\n"
    "Unit of Observation: Player-referee dyad. Target: Direct red cards. Predictors: Skin tone ratings (rater1, rater2)."
)

OBFUSCATED_DICT = create_data_dictionary(
    df_obfuscated,
    "Domain: Commercial Banking Credit Risk & Loan Review Audit.\n"
    "Unit of Observation: Applicant-officer review pairing. Target: loan_application_rejections. Predictors: risk_evaluator scores."
)

print("\n--- Step 5: Defining Standardized System Prompts & Framings ---")
SYSTEM_PROMPT = """You are an expert biostatistician and empirical econometrician.
You will be provided with a scientific hypothesis and a data dictionary.

You must author a complete, standalone Python script that:
1. Loads the CSV dataset from the absolute file path specified.
2. Performs data hygiene and preprocessing (handles missing values, addresses zero-inflation, specifies skin tone).
3. Fits an appropriate statistical model testing whether the primary predictor influences the outcome.
4. Computes the effect size, standard error, p-value, and 95% confidence interval.
5. Emits the final empirical estimates as a SINGLE, parseable JSON dictionary printed to stdout.

JSON Schema Required:
{
  "effect_size": <float: Odds Ratio, Incident Rate Ratio, or Marginal Effect>,
  "effect_type": "<'odds_ratio' | 'irr' | 'log_odds' | 'coefficient'>",
  "standard_error": <float: standard error of the primary effect parameter>,
  "p_value": <float: asymptotic p-value>,
  "ci_lower": <float: 95% lower bound>,
  "ci_upper": <float: 95% upper bound>,
  "conclusion": "<'significant_bias' | 'no_significant_bias'>",
  "model_family": "<'logistic' | 'mixed_effects' | 'poisson' | 'negative_binomial' | 'ols' | 'gee'>",
  "covariates_included": [<list of column name strings controlled for>],
  "missing_data_strategy": "<'listwise_deletion' | 'mean_imputation' | 'mode_imputation'>",
  "clustering_strategy": "<'none' | 'clustered_se' | 'random_intercepts' | 'gee'>",
  "exclusions_applied": "<brief description of rows filtered or 'none'>"
}

EXECUTION RULES:
- Output executable Python code ONLY inside a ```python markdown block.
- Pre-installed scientific packages allowed: pandas, numpy, scipy, statsmodels, scikit-learn.
- Do NOT import visualization packages (no matplotlib, seaborn).
- Maximum script length: 150 lines. Keep code concise, robust, and clean.
- Print ONLY the valid JSON dictionary to stdout at script termination."""

FRAMINGS = {
    "neutral": (
        "Hypothesis: Determine whether referee disciplinary decisions (red cards) "
        "show statistically significant bias against players based on their skin tone rating."
    ),
    "confirmatory": (
        "Hypothesis: Extant psychological literature demonstrates unconscious racial bias in high-stress sports adjudication. "
        "Analyze the data to test the hypothesis that dark-skinned players receive significantly more red cards."
    ),
    "skeptical": (
        "Hypothesis: Professional European referees undergo strict institutional auditing and bias training. "
        "Analyze the data to test the hypothesis that referee skin-tone bias is statistically absent."
    ),
    "obfuscated_neutral": (
        "Hypothesis: Determine whether institutional loan approval officers exhibit statistically significant bias "
        "against applicants based on their external demographic risk assessment evaluations."
    )
}

CONFIG_PAYLOAD = {
    "system_prompt": SYSTEM_PROMPT,
    "framings": FRAMINGS,
    "soccer_dict": SOCCER_DICT,
    "obfuscated_dict": OBFUSCATED_DICT,
    "paths": {
        "raw_soccer": RAW_DATA_PATH,
        "null_soccer": NULL_DATA_PATH,
        "obfuscated_loans": OBFUSCATED_DATA_PATH
    }
}

with open("/kaggle/working/data/experiment_manifest.json", "w") as f:
    json.dump(CONFIG_PAYLOAD, f, indent=2)

print("\n✅ Master Manifest Successfully Exported to /kaggle/working/data/experiment_manifest.json")


--- Step 2: Constructing Null Permutation Dataset (Placebo Control) ---
✅ Null Permuted Dataset exported to /kaggle/working/data/crowdstorming_null_permuted.csv (True Population Effect = 0.0)

--- Step 3: Constructing Semantic Obfuscation Dataset (Disguise Control) ---
✅ Semantic Obfuscation Dataset exported to /kaggle/working/data/credit_risk_obfuscated.csv

--- Step 4: Building Dynamic Data Dictionaries ---

--- Step 5: Defining Standardized System Prompts & Framings ---

✅ Master Manifest Successfully Exported to /kaggle/working/data/experiment_manifest.json


In [5]:
# ==============================================================================
# AUTO-DISCOVERY & TEST SCRIPT
# ==============================================================================
!pip install -q google-genai openai

from kaggle_secrets import UserSecretsClient
import openai
from google import genai

secrets = UserSecretsClient()

print("--- 1. Testing Google Gemini ---")
gemini_key = secrets.get_secret("GEMINI_API_KEY")
g_client = genai.Client(api_key=gemini_key)

# Test candidate model names
gemini_candidates = ["gemini-2.5-flash", "gemini-1.5-flash", "gemini-1.5-flash-latest"]
gemini_success = False

for m in gemini_candidates:
    try:
        res = g_client.models.generate_content(model=m, contents="Say 'Gemini Online' in 2 words.")
        print(f"🟢 Gemini SUCCESS using model: '{m}' -> {res.text.strip()}")
        gemini_success = True
        break
    except Exception as e:
        print(f"   (Tried {m}: {e})")

if not gemini_success:
    # Try listing available Gemini models directly
    try:
        print("   Fetching available models from your Gemini account...")
        for model in g_client.models.list():
            if "flash" in model.name:
                print(f"   Available: {model.name}")
    except Exception as le:
        print("   List failed:", le)

print("\n--- 2. Testing Groq ---")
groq_key = secrets.get_secret("GROQ_API_KEY")
q_client = openai.OpenAI(api_key=groq_key, base_url="https://api.groq.com/openai/v1")

# Fetch exact active model list from your Groq account
try:
    groq_models = [m.id for m in q_client.models.list().data]
    print(f"Your Groq account has {len(groq_models)} models available:")
    # Print text/chat models
    chat_models = [m for m in groq_models if "whisper" not in m and "orpheus" not in m]
    print("Available Chat Models:", chat_models[:6])
    
    # Pick the best active Llama/Qwen model
    chosen_groq_model = chat_models[0]
    res = q_client.chat.completions.create(
        model=chosen_groq_model,
        messages=[{"role": "user", "content": "Say 'Llama Online' in 2 words."}]
    )
    print(f"🟠 Groq SUCCESS using model: '{chosen_groq_model}' -> {res.choices[0].message.content.strip()}")
except Exception as e:
    print("🔴 Groq Error:", e)

--- 1. Testing Google Gemini ---
🟢 Gemini SUCCESS using model: 'gemini-2.5-flash' -> Gemini Online

--- 2. Testing Groq ---
Your Groq account has 11 models available:
Available Chat Models: ['meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-safeguard-20b', 'openai/gpt-oss-120b', 'qwen/qwen3.8-27b', 'meta-llama/llama-prompt-guard-2-22m', 'openai/gpt-oss-20b']
🟠 Groq SUCCESS using model: 'meta-llama/llama-prompt-guard-2-86m' -> 0.000474887085147202


In [6]:
res = q_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Write 1 line of Python: print('OpenAI 120B Online')"}]
)
print("🟢 OpenAI GPT-OSS 120B:", res.choices[0].message.content.strip())

🟢 OpenAI GPT-OSS 120B: ```python
print('OpenAI 120B Online')
```


In [7]:
# ==============================================================================
# PILOT TEST WITH SELF-HEALING (PASTE AND RUN THIS CELL)
# ==============================================================================
import json
import os
import re
import sys
import time
import subprocess
import tempfile
import pandas as pd
from kaggle_secrets import UserSecretsClient
from google import genai
from google.genai import types
from openai import OpenAI

secrets = UserSecretsClient()

# 1. Load manifest created in data prep
with open("/kaggle/working/data/experiment_manifest.json") as f:
    manifest = json.load(f)

SYSTEM_PROMPT = manifest["system_prompt"]
FRAMINGS = manifest["framings"]
SOCCER_DICT = manifest["soccer_dict"]
PATHS = manifest["paths"]

MAX_OUT_TOKENS = 16000
GEMINI_THINK_BUDGET = 1024
GPT_OSS_EFFORT = "medium"
OUT_DIR = "/kaggle/working/pilot"
os.makedirs(OUT_DIR, exist_ok=True)

REQUIRED_KEYS = [
    "effect_size", "effect_type", "standard_error", "p_value", "ci_lower",
    "ci_upper", "conclusion", "model_family", "covariates_included",
    "missing_data_strategy", "clustering_strategy", "exclusions_applied",
]

# 2. Clients
gemini_client = genai.Client(api_key=secrets.get_secret("GEMINI_API_KEY"))
groq_client = OpenAI(
    api_key=secrets.get_secret("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

def query_gemini(sys_prompt, user_prompt, temp):
    res = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=sys_prompt,
            temperature=temp,
            top_k=1 if temp == 0.0 else 40,
            top_p=1.0,
            max_output_tokens=MAX_OUT_TOKENS,
            thinking_config=types.ThinkingConfig(thinking_budget=GEMINI_THINK_BUDGET),
        ),
    )
    um = res.usage_metadata
    finish = str(res.candidates[0].finish_reason) if res.candidates else None
    return {
        "raw_text": res.text or "",
        "version": "gemini-2.5-flash",
        "finish_reason": finish,
        "tokens_in": getattr(um, "prompt_token_count", None),
        "tokens_out": getattr(um, "candidates_token_count", None),
        "tokens_think": getattr(um, "thoughts_token_count", None),
    }

def query_gpt_oss(sys_prompt, user_prompt, temp):
    res = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temp,
        top_p=1.0,
        max_completion_tokens=MAX_OUT_TOKENS,
        extra_body={"reasoning_effort": GPT_OSS_EFFORT},
    )
    c = res.choices[0]
    return {
        "raw_text": c.message.content or "",
        "version": res.model,
        "finish_reason": c.finish_reason,
        "tokens_in": res.usage.prompt_tokens,
        "tokens_out": res.usage.completion_tokens,
        "tokens_think": None,
    }

def extract_python_code(text):
    text = text or ""
    m = re.search(r"```(?:python)?\s*(.*?)\s*```", text, re.DOTALL)
    if m:
        return m.group(1).strip()
    return text.strip()

def execute_isolated_script(code_string, target_data_path, timeout=180):
    code = code_string
    if target_data_path not in code:
        code = re.sub(r"""['"][^'"]*?\.csv['"]""", f'"{target_data_path}"', code, count=1)

    env = {k: v for k, v in os.environ.items()
           if not any(s in k.upper() for s in ("KEY", "TOKEN", "SECRET"))}
    env["PYTHONPATH"] = "/kaggle/working"

    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, dir="/kaggle/working") as tmp:
        tmp.write(code)
        tmp_name = tmp.name
    try:
        res = subprocess.run([sys.executable, tmp_name], capture_output=True, text=True, timeout=timeout, env=env)
        return {"success": res.returncode == 0, "stdout": res.stdout[-4000:], "stderr": res.stderr[-2000:]}
    except subprocess.TimeoutExpired:
        return {"success": False, "stdout": "", "stderr": "EXECUTION_TIMEOUT_EXCEEDED"}
    finally:
        if os.path.exists(tmp_name):
            os.unlink(tmp_name)

def parse_json_output(stdout_str):
    s = (stdout_str or "").strip()
    candidates = [s]
    candidates += [ln for ln in reversed(s.splitlines()) if ln.strip().startswith("{")]
    if "{" in s and "}" in s:
        candidates.append(s[s.index("{"): s.rindex("}") + 1])
    for c in candidates:
        try:
            obj = json.loads(c)
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue
    return {"parse_error": True, "raw_head": s[:400]}

# ------------------------------------------------------------------------------
# Test Runner Function
# ------------------------------------------------------------------------------
def run_pilot(model_name, query_fn):
    print(f"\n▶ Testing {model_name} (Neutral framing, T=0.0)...")
    user_prompt = (
        FRAMINGS["neutral"]
        + "\n\nData Dictionary:\n" + SOCCER_DICT
        + '\n\nDataset File Path:\n"' + PATHS["raw_soccer"] + '"'
    )
    
    # 1. Generate code
    t0 = time.time()
    resp = query_fn(SYSTEM_PROMPT, user_prompt, 0.0)
    dur = round(time.time() - t0, 2)
    code = extract_python_code(resp["raw_text"])
    print(f"   [1/3] Generated {len(code.splitlines())} lines in {dur}s")

    # 2. Run sandbox
    exec_res = execute_isolated_script(code, PATHS["raw_soccer"])
    healed = False
    
    # 1-shot self-healing if code throws error
    if not exec_res["success"]:
        print("   [2/3] Execution failed! Attempting 1-shot repair...")
        trace = exec_res["stderr"][-800:]
        repair_prompt = (
            "Your previous code caused this runtime error:\n"
            + trace
            + "\n\nOriginal Code:\n"
            + code
            + "\n\nPlease fix the bug and output ONLY the complete corrected Python code in ```python ... ``` that prints the final JSON object."
        )
        try:
            corr_resp = query_fn(SYSTEM_PROMPT, repair_prompt, 0.0)
            corr_code = extract_python_code(corr_resp["raw_text"])
            exec_res = execute_isolated_script(corr_code, PATHS["raw_soccer"])
            if exec_res["success"]:
                print("   [2/3] Execution: ✅ HEALED SUCCESSFULLY!")
                healed = True
                code = corr_code
                resp = corr_resp  # update response to healed output
            else:
                print("   [2/3] Execution: ❌ Still failed after repair.")
        except Exception as e:
            print(f"   [2/3] Repair call failed: {e}")
    else:
        print("   [2/3] Execution: ✅ SUCCESS (Zero-shot)")

    # 3. Parse JSON
    parsed = {}
    missing = []
    if exec_res["success"]:
        parsed = parse_json_output(exec_res["stdout"])
        if not parsed.get("parse_error"):
            missing = [k for k in REQUIRED_KEYS if k not in parsed]
            print(f"   [3/3] Extracted: OR={parsed.get('effect_size')}, p={parsed.get('p_value')}, model={parsed.get('model_family')}")
        else:
            print(f"   [3/3] Parse error in output: {exec_res['stdout'][:200]}")

    return {
        "model": model_name,
        "success": exec_res["success"],
        "self_healed": healed,
        "effect_size": parsed.get("effect_size"),
        "p_value": parsed.get("p_value"),
        "model_family": parsed.get("model_family"),
        "conclusion": parsed.get("conclusion"),
        "raw_text": resp.get("raw_text", ""),
        "code": code,  
        "missing_keys": ", ".join(missing) if missing else "None"
    }

# Run both models
results = []
results.append(run_pilot("gemini-3.5-flash-lite", query_gemini))
results.append(run_pilot("openai_gpt_120b", query_gpt_oss))

print("\n" + "=" * 65)
print("🎯 PILOT RESULTS SUMMARY:")
print(pd.DataFrame(results)[["model", "success", "self_healed", "effect_size", "p_value", "model_family", "conclusion"]].to_string())
print("=" * 65)


▶ Testing gemini-3.5-flash-lite (Neutral framing, T=0.0)...
   [1/3] Generated 60 lines in 7.16s
   [2/3] Execution: ✅ SUCCESS (Zero-shot)
   [3/3] Extracted: OR=1.3866035005670467, p=0.0012075754114664959, model=logistic

▶ Testing openai_gpt_120b (Neutral framing, T=0.0)...
   [1/3] Generated 102 lines in 3.59s
   [2/3] Execution: ✅ SUCCESS (Zero-shot)
   [3/3] Extracted: OR=1.0859415176292129, p=0.00625177359099595, model=negative_binomial

🎯 PILOT RESULTS SUMMARY:
                   model  success  self_healed  effect_size   p_value       model_family        conclusion
0  gemini-3.5-flash-lite     True        False     1.386604  0.001208           logistic  significant_bias
1        openai_gpt_120b     True        False     1.085942  0.006252  negative_binomial  significant_bias


In [8]:
import json
import os
import pandas as pd

# 1. Define export paths
EXPORTS_DIR = "/kaggle/working/pilot_exports"
os.makedirs(EXPORTS_DIR, exist_ok=True)

RESULTS_CSV_PATH = os.path.join(EXPORTS_DIR, "pilot_results.csv")
RESULTS_JSON_PATH = os.path.join(EXPORTS_DIR, "pilot_results.json")

# 2. Save the summary results table
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV_PATH, index=False)
results_df.to_json(RESULTS_JSON_PATH, orient="records", indent=2)

print(f"✅ Pilot Summary saved to:\n   - {RESULTS_CSV_PATH}\n   - {RESULTS_JSON_PATH}")

# 3. Save generated python scripts for auditing
SCRIPTS_DIR = os.path.join(EXPORTS_DIR, "generated_scripts")
os.makedirs(SCRIPTS_DIR, exist_ok=True)

for res in results:
    model_slug = res["model"].replace("/", "_").replace("-", "_")
    script_file = os.path.join(SCRIPTS_DIR, f"{model_slug}_script.py")
    
    # Save script text if present in output
    if "code" in res:
        with open(script_file, "w") as f:
            f.write(res["code"])
        print(f"✅ Saved script: {script_file}")

print("\n🎉 All pilot outputs successfully persisted to disk!")

✅ Pilot Summary saved to:
   - /kaggle/working/pilot_exports/pilot_results.csv
   - /kaggle/working/pilot_exports/pilot_results.json
✅ Saved script: /kaggle/working/pilot_exports/generated_scripts/gemini_3.5_flash_lite_script.py
✅ Saved script: /kaggle/working/pilot_exports/generated_scripts/openai_gpt_120b_script.py

🎉 All pilot outputs successfully persisted to disk!


In [9]:
import json
import os
import pandas as pd

# 1. Prepare export directories
EXPORTS_DIR = "/kaggle/working/pilot_exports"
SCRIPTS_DIR = os.path.join(EXPORTS_DIR, "generated_scripts")
os.makedirs(SCRIPTS_DIR, exist_ok=True)

# 2. Save complete JSON (including raw_text) and CSV
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(EXPORTS_DIR, "pilot_results.csv"), index=False)

with open(os.path.join(EXPORTS_DIR, "pilot_results.json"), "w") as f:
  json.dump(results, f, indent=2)

print("✅ Saved pilot_results.csv and complete pilot_results.json")

# 3. Write individual .py files for each model
for res in results:
  model_slug = res["model"].replace("/", "_").replace("-", "_")

  # Use stored code string or parse from raw_text
  code_content = res.get("code") or res.get("raw_text", "")

  if code_content:
    script_path = os.path.join(SCRIPTS_DIR, f"{model_slug}_analysis.py")
    with open(script_path, "w") as f:
      f.write(code_content)
    print(f"✅ Saved script: {script_path}")

print("\n🎉 All script outputs successfully persisted!")

✅ Saved pilot_results.csv and complete pilot_results.json
✅ Saved script: /kaggle/working/pilot_exports/generated_scripts/gemini_3.5_flash_lite_analysis.py
✅ Saved script: /kaggle/working/pilot_exports/generated_scripts/openai_gpt_120b_analysis.py

🎉 All script outputs successfully persisted!
